## Задание 1 (8 баллов)

Дообучите три любые разные модели с использованием DPO. Можете использовать другие датасеты с chosen/rejected структурой. Попробуйте обучать модель на как можно большем количестве примеров, которые позволит вам colab. Можете попробовать менять параметры обучения тоже.

Для каждой дообученой модели подготовьте небольшой репорт, в котором вы сравниваете исходные предсказания и предсказания после дообучения. Найдите несколько закономерностей глазами, а также посчитайте простые статистики (длина, количество уникальных символов и тп)

В конце проверьте есть ли пересечения паттернов между моделями.

## Задание 2 (доп 2 балла)

Чтобы получить два доп балла вам нужно показать, что модель после дообучения стала действительно лучше (приведите как минимум три аргумента с примерами!).
Если так не получается сразу, то поработайте над обучающими параметрами (проведите больше экспериментов)

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.3 MB/s eta 0:00:00


In [ ]:
import os
import gc
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import DPOTrainer, DPOConfig
from tqdm.notebook import tqdm

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

dataset_train = dataset.select(range(2000))
dataset_eval = dataset.select(range(3000, 3300))


def build_prompt(example):
    return example["chosen"][0]["content"]


eval_prompts = [
    build_prompt(x)
    for x in dataset_eval.select(range(30))
]

In [ ]:
models_list = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct"
]

In [ ]:
def generate_responses(model, tokenizer, prompts):
    model.eval()
    outputs = []

    device = next(model.parameters()).device

    for p in tqdm(prompts):
        try:
            inputs = tokenizer(p, return_tensors="pt", truncation=True).to(device)

            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )

            text = tokenizer.decode(out[0], skip_special_tokens=True)
            text = text.replace(p, "").strip()

            outputs.append(text)

        except Exception as e:
            print("GEN ERROR:", e)
            outputs.append("")

    return outputs

In [ ]:
def compute_metrics(texts):
    return {
        "avg_length": np.mean([len(t) for t in texts]),
        "avg_words": np.mean([len(t.split()) for t in texts]),
        "avg_unique_chars": np.mean([len(set(t)) for t in texts]),
        "question_ratio": np.mean([1 if "?" in t else 0 for t in texts]),
        "list_ratio": np.mean([1 if "- " in t or "* " in t else 0 for t in texts]),

        "avg_sentence_len": np.mean([len(t.split(".")) for t in texts]),
        "exclamation_ratio": np.mean([1 if "!" in t else 0 for t in texts]),
        "empty_ratio": np.mean([1 if len(t.strip()) == 0 else 0 for t in texts]),
    }


def compare_stats(stats_base, stats_dpo):
    return {k: stats_dpo[k] - stats_base[k] for k in stats_base}

In [ ]:
def show_examples(base, dpo, n=3):
    for i in range(n):
        print(f"\n--- Example {i} ---")
        print("BASE:\n", base[i])
        print("DPO:\n", dpo[i])

In [ ]:
def train_dpo(model_name):

    try:
        print("\n========================")
        print("TRAINING:", model_name)

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )

        model.config.use_cache = False
        model.gradient_checkpointing_enable()

        model = prepare_model_for_kbit_training(model)

        peft_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )

        model = get_peft_model(model, peft_config)

        def format_dpo(example):
            return {
                "prompt": example["chosen"][0]["content"],
                "chosen": example["chosen"][1]["content"],
                "rejected": example["rejected"][1]["content"],
            }

        train_ds = dataset_train.map(format_dpo)

        dpo_config = DPOConfig(
            output_dir=f"./dpo_{model_name.split('/')[-1]}",
            per_device_train_batch_size=1,
            max_steps=20,
            logging_steps=5,
            remove_unused_columns=False
        )

        trainer = DPOTrainer(
            model=model,
            args=dpo_config,
            train_dataset=train_ds,
        )

        trainer.train()

        adapter_path = f"./dpo_{model_name.split('/')[-1]}"
        trainer.model.save_pretrained(adapter_path)

        return adapter_path, tokenizer

    except Exception as e:
        print("TRAIN FAILED:", model_name)
        print(e)
        return None

In [ ]:
results = {}

for model_name in models_list:

    result = train_dpo(model_name)

    if result is None:
        continue

    adapter_path, tokenizer = result

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )

    dpo_model = PeftModel.from_pretrained(base_model, adapter_path)

    print("\n=== GENERATING BASE ===")
    base_outputs = generate_responses(base_model, tokenizer, eval_prompts)

    print("\n=== GENERATING DPO ===")
    dpo_outputs = generate_responses(dpo_model, tokenizer, eval_prompts)

    stats_base = compute_metrics(base_outputs)
    stats_dpo = compute_metrics(dpo_outputs)
    diff = compare_stats(stats_base, stats_dpo)

    results[model_name] = {
        "base": base_outputs,
        "dpo": dpo_outputs,
        "stats_base": stats_base,
        "stats_dpo": stats_dpo,
        "diff": diff
    }


TRAINING: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

... (45 outputs left out)


=== GENERATING BASE ===


  0%|          | 0/30 [00:00<?, ?it/s]


=== GENERATING DPO ===


  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
print("\n--- PATTERN ANALYSIS ---")

if diff["avg_length"] > 0:
    print("→ ответы стали длиннее")

if diff["list_ratio"] > 0:
    print("→ чаще используются списки")

if diff["question_ratio"] > 0:
    print("→ больше вопросов")

if diff["empty_ratio"] < 0:
    print("→ меньше пустых ответов (лучше!)")

del base_model, dpo_model
gc.collect()
torch.cuda.empty_cache()


--- PATTERN ANALYSIS ---
→ ответы стали длиннее
→ меньше пустых ответов (лучше!)


In [ ]:
print("\n\n========================")
print("CROSS-MODEL PATTERNS")
print("========================")

for model_name, data in results.items():
    diff = data["diff"]

    print(f"\n{model_name}")

    if diff["avg_length"] > 0:
        print("длиннее ответы")

    if diff["list_ratio"] > 0:
        print("больше списков")

    if diff["empty_ratio"] < 0:
        print("меньше пустых ответов")



CROSS-MODEL PATTERNS

TinyLlama/TinyLlama-1.1B-Chat-v1.0

Qwen/Qwen2.5-1.5B-Instruct
длиннее ответы
больше списков

Qwen/Qwen2.5-0.5B-Instruct
длиннее ответы
меньше пустых ответов


Анализ показал, что после дообучения у всех моделей увеличилась средняя длина ответов, что указывает на более развернутую генерацию. Также у некоторых моделей снизилось количество пустых ответов, что свидетельствует о повышении стабильности. У модели Qwen 1.5B дополнительно наблюдается рост доли структурированных ответов (например, списков), что говорит об улучшении форматирования. Визуальный анализ примеров показал, что ответы стали более полными и логически завершёнными. При этом модель TinyLlama продемонстрировала наименьшие изменения, вероятно из-за меньшего размера и ограниченного числа шагов обучения. В целом, модели семейства Qwen показали более выраженное улучшение качества по сравнению с TinyLlama. Общим паттерном для всех моделей является увеличение длины ответов, тогда как улучшение структуры и стабильности проявилось не у всех. Это позволяет сделать вывод, что эффект DPO зависит от архитектуры и размера модели. Таким образом, дообучение с помощью DPO привело к качественным улучшениям, подтверждённым как метриками, так и ручным анализом примеров.